# Introduction

What is Conformal Prediction ?

Conformal prediction is a new way we can construct the new prediction set by setting the quantile threshold, so we could use this for Data calibration to see how far the LLM can be evaluate the uncerntainty.

In [1]:
import numpy as np

from collections import Counter

In [2]:
np.random.seed(42)

In [14]:
# Options QCM
K = 6
options_labels = ['A', 'B', 'C', 'D', 'E', 'F']

# number of calibration and test instances
n_cal = 500
n_test = 500

In [15]:
# Generate random logies
def softmax(logits):
    exp_logits = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    return exp_logits / exp_logits.sum(axis=1, keepdims=True)

In [16]:
cal_logits = np.random.randn(int(n_cal), K) * 2
test_logits = np.random.randn(int(n_test), K) *2 

In [17]:
cal_softmax = softmax(cal_logits)
test_softmax = softmax(test_logits)


In [18]:
print("Calibration probs shape:", cal_softmax.shape)
print("Test probs shape:", test_softmax.shape)
for i, label in enumerate(options_labels):
    print(f"  {label}: {cal_softmax[0][i]:.4f}")

Calibration probs shape: (500, 6)
Test probs shape: (500, 6)
  A: 0.0157
  B: 0.0412
  C: 0.0221
  D: 0.0487
  E: 0.0949
  F: 0.7773


In [20]:
cal_true = np.random.randint(0, 4, size=n_cal) # 0=A, 1=B, 2=C, 3=D
test_true = np.random.randint(0, 4, size=n_test)

print("First 10 calibration true labels", [options_labels[t] for t in cal_true[:10]])
print("First 10 test true label", [options_labels[t] for t in test_true[:10]])
print("\nDistribution of true labels (cal):", Counter([options_labels[t] for t in cal_true]))


First 10 calibration true labels ['B', 'A', 'B', 'D', 'C', 'B', 'A', 'C', 'B', 'B']
First 10 test true label ['C', 'C', 'B', 'A', 'B', 'A', 'B', 'D', 'C', 'A']

Distribution of true labels (cal): Counter({'B': 134, 'A': 131, 'D': 128, 'C': 107})


In [22]:
def lac_score(probs, true_labels):
    return 1.0 - probs[true_labels]


example_probs = cal_softmax[0]
example_true = cal_true[0]
example_score = lac_score(example_probs, example_true)
example_score

np.float64(0.9587504478192324)

In [25]:
def aps_score(probs, true_label):
    true_prob = probs[true_label]

    score = np.sum(probs[probs >= true_prob])

    return score

In [27]:
example_score_aps = aps_score(example_probs, example_true)
example_score_aps

np.float64(0.9621651435764376)

In [28]:
# Show the ranking
sorted_indices = np.argsort(example_probs)[::-1]  # highest to lowest
print("Ranked probabilities:")
cumsum = 0
for rank, idx in enumerate(sorted_indices):
    cumsum += example_probs[idx]
    marker = " <-- TRUE LABEL" if idx == example_true else ""
    print(f"  Rank {rank+1}: {options_labels[idx]} = {example_probs[idx]:.4f} (cumsum: {cumsum:.4f}){marker}")
    if idx == example_true:
        break

print(f"\nAPS score: {example_score_aps:.4f}")

Ranked probabilities:
  Rank 1: F = 0.7773 (cumsum: 0.7773)
  Rank 2: E = 0.0949 (cumsum: 0.8722)
  Rank 3: D = 0.0487 (cumsum: 0.9209)
  Rank 4: B = 0.0412 (cumsum: 0.9622) <-- TRUE LABEL

APS score: 0.9622


In [29]:
def compute_calibration_score(probs, true_labels, score_fn):
    scores = []
    for i in range(len(true_labels)):
        s = score_fn(probs[i], true_labels[i])
        scores.append(s)

    return np.array(scores)

In [30]:
cal_scores_lac = compute_calibration_score(cal_softmax, cal_true, lac_score)
cal_scores_aps = compute_calibration_score(cal_softmax, cal_true, aps_score)

print("LAC calibration scores:")
print(f"  Min: {cal_scores_lac.min():.4f}")
print(f"  Max: {cal_scores_lac.max():.4f}")
print(f"  Mean: {cal_scores_lac.mean():.4f}")
print(f"  First 10: {[f'{s:.3f}' for s in cal_scores_lac[:10]]}")

print(f"\nAPS calibration scores:")
print(f"  Min: {cal_scores_aps.min():.4f}")
print(f"  Max: {cal_scores_aps.max():.4f}")
print(f"  Mean: {cal_scores_aps.mean():.4f}")
print(f"  First 10: {[f'{s:.3f}' for s in cal_scores_aps[:10]]}")

LAC calibration scores:
  Min: 0.0095
  Max: 0.9999
  Mean: 0.8314
  First 10: ['0.959', '0.962', '0.904', '0.986', '0.203', '0.906', '0.835', '0.988', '0.994', '0.913']

APS calibration scores:
  Min: 0.2726
  Max: 1.0000
  Mean: 0.8842
  First 10: ['0.962', '0.983', '0.938', '0.981', '0.797', '0.942', '0.697', '0.982', '0.998', '0.939']


In [33]:
def compute_threshold(cal_scores, alpha):
    n = len(cal_scores)
    quantile_level = np.ceil((n+1)*(1-alpha))/n
    quantile_level = min(quantile_level, 1.0)

    q_hat = np.quantile(cal_scores, quantile_level, method='higher')

    return q_hat

In [34]:
alpha = 0.1
q_hat_lac = compute_threshold(cal_scores_lac, alpha)
q_hat_aps = compute_threshold(cal_scores_aps, alpha)

n = len(cal_scores_lac)
quantile_level = np.ceil((n + 1) * (1 - alpha)) / n

print(f"Error rate = {alpha}")
print(f"Target coverage = {1 - alpha}")
print(f"n (calibration size) = {n}")
print(f"Quantile level = ceil(({n}+1)*{1-alpha}) / {n} = {quantile_level:.4f}")
print(f"\nLAC threshold q_hat = {q_hat_lac:.4f}")
print(f"  Meaning: include option Y' if f(X)_Y' >= {1 - q_hat_lac:.4f}")
print(f"\nAPS threshold q_hat = {q_hat_aps:.4f}")
print(f"  Meaning: include option Y' if cumulative ranked prob <= {q_hat_aps:.4f}")

Error rate = 0.1
Target coverage = 0.9
n (calibration size) = 500
Quantile level = ceil((500+1)*0.9) / 500 = 0.9020

LAC threshold q_hat = 0.9973
  Meaning: include option Y' if f(X)_Y' >= 0.0027

APS threshold q_hat = 1.0000
  Meaning: include option Y' if cumulative ranked prob <= 1.0000


In [36]:
def build_prediction_set_lac(probs, q_hat):
    prediction_set = []
    for y_new in range(len(probs)):
        score = 1 - probs[y_new]
        if score <= q_hat:
            prediction_set.append(y_new)
    
    if len(prediction_set) == 0:
        prediction_set = [np.argmax(probs)]

    return prediction_set


def build_prediction_set_aps(probs, q_hat):
    prediction_set = []
    for y_new in range(len(probs)):
        score = np.sum(probs[probs >= probs[y_new]])
        if score <= q_hat:
            prediction_set.append(score)

    if len(prediction_set) == 0:
        prediction_set = [np.argmax(probs)]

    return prediction_set
        

In [39]:
print(f"\nProbabilities: {[f'{p:.4f}' for p in test_softmax[0]]}")



Probabilities: ['0.2760', '0.5111', '0.1619', '0.0012', '0.0438', '0.0059']


In [40]:
print(f"\n--- LAC (threshold = {q_hat_lac:.4f}) ---")
for y in range(K):
    score = 1.0 - test_softmax[0][y]
    included = "YES" if score <= q_hat_lac else "NO"
    print(f"  {options_labels[y]}: prob={test_softmax[0][y]:.4f}, score={score:.4f}, included={included}")


--- LAC (threshold = 0.9973) ---
  A: prob=0.2760, score=0.7240, included=YES
  B: prob=0.5111, score=0.4889, included=YES
  C: prob=0.1619, score=0.8381, included=YES
  D: prob=0.0012, score=0.9988, included=NO
  E: prob=0.0438, score=0.9562, included=YES
  F: prob=0.0059, score=0.9941, included=YES


In [41]:
pred_set_lac = build_prediction_set_lac(
    test_softmax[0]
    , q_hat_lac
)
print(f"\nLAC prediction set: {{{', '.join([options_labels[i] for i in pred_set_lac])}}}")
print(f"Set size: {len(pred_set_lac)}")
print(f"True label in set: {test_true[0] in pred_set_lac}")


LAC prediction set: {A, B, C, E, F}
Set size: 5
True label in set: True
